In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob

plt.rcParams.update({
    "font.size" : 15,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "stix",
})

In [ ]:
c_s = 1.0 / np.sqrt(3.0)
f0 = 0.05
rSphere = 20
u0 = 0.01
rho0 = 1.0

# vis = 0.0               # tau=0.5
# vis = 0.033333333       # tau=0.6
vis = 0.166666667       # tau=1.0

step = 240

folderVec = [
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/acoustic_validation/pulsatingSphere_srt_tau05/probes/probeX",
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/acoustic_validation/pulsatingSphere_srt_tau06/probes/probeX",
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/acoustic_validation/pulsatingSphere_srt_tau1/probes/probeX",
]

labelVec = [
    'LBM BGK',
]

colorVec = [
    # "#910000",
    # "#119100",
    "#001AFF",
    '#FF5733',
    "#B700FF",
]

lstyleVec = [
    # '--',
    # '-.',
    ':',
    '--'
]

In [ ]:


tau = vis / c_s**2 + 0.5
visBulk = 2.0 / 3.0 * vis
tau_vi = 1.0 / c_s**2 * (4.0 / 3.0 * vis + visBulk)

omega0 = 2.0 * np.pi * f0
k0 = omega0 / c_s

omega = omega0 * (1.0 - 1.0 / 8.0 * (omega0 * tau_vi)**2)
# alpha_t = omega0**2 * tau_vi / 2.0
k = k0 / (1.0 + 3.0 / 8.0 * (omega0 * tau_vi)**2)
alpha_x = omega0 * tau_vi * k0 * 0.5
k_hat = k - 1j * alpha_x

print(f"c_s = {c_s:.6f}")
print(f"f0 = {f0:6f}")
print(f"vis_bulk = {visBulk:.6f}")
print(f"tau = {tau:.6f}")
print(f"tau_vi = {tau_vi:.6f}")
print(f"omega0 = {omega0:.6f}")
print(f"alpha_x = {alpha_x:.6f}")
print(f"k = {k:.6f}")

Q = 4.0 * np.pi * rSphere**2 * rho0 * u0


In [ ]:
def read_col_probe(filePath, colIdx, skipHeader=2):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, skip_header=skipHeader, usecols=colIdx)
        vec = vec[~np.isnan(vec)]
    f.close()
    return vec
def get_probe_coord_value(filePath, coordIdx):
    with open(filePath, 'r') as f:
        header = f.readline()
        coordValue = float(header.split('(')[1].split(')')[0].split(',')[coordIdx])
    return coordValue
def read_csv_col(filePath, colIdx, skipHeader=1):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, delimiter=',', skip_header=skipHeader, usecols=colIdx)
    f.close()
    return vec
    

coordMap = { 'x': 0, 'y': 1, 'z': 2 }
coordIdx = coordMap['x']
colIdx = 2

coordVecVec = []
dataVecVec = []
for case in range(0, len(folderVec)):
    fileList = sorted(glob.glob(f"{folderVec[case]}/probe*.txt"))
    coordVec = []
    dataVec = []
    for filePath in fileList:
        coordValue = get_probe_coord_value(filePath, coordIdx)
        # coordVec.append(coordValue)
        coordVec.append(coordValue + 0.5)
        stepCol = read_col_probe(filePath, 0, 2)
        dataCol = read_col_probe(filePath, colIdx)
        targetStepIdx = int(np.abs(stepCol - (step + 1)).argmin())
        dataVec.append(dataCol[targetStepIdx]-rho0)
    coordVecVec.append(coordVec)
    dataVecVec.append(dataVec)


In [ ]:
x = np.linspace(rSphere + 1.0, 150.0, 300)
#! book:
# pPrime = 1j * omega * Q / (4.0 * np.pi * abs(xSafe)) * np.exp(1j * (omega0 * step - k_hat * abs(xSafe)))
#! claude opus 4.6 correction:
pPrime = 1j * omega0 * rho0 * u0 * rSphere**2 / (x * (1.0 + 1j * k_hat * rSphere))
pPrime = pPrime * np.exp(1j * (omega0 * step - k_hat * (x - rSphere)))
rhoPrime = np.real(pPrime) / c_s**2

In [ ]:
fig1, ax1 = plt.subplots(1, 1, figsize=(12, 3), facecolor='w', edgecolor='w')
ax1.plot(x, np.real(rhoPrime), c='k', linestyle='-', lw=2, label='Analytical')

for case in range(0, len(folderVec)):
    ax1.plot(coordVecVec[case], dataVecVec[case], c=colorVec[case], marker='o', ms=6, fillstyle='none', lw=0, label=labelVec[case])

# tau=0.5 compare with ParaView extraction
# pvDataX = read_csv_col("./pulsatingSphere_tau05_step200_pv.csv", 0)
# pvdataRho = read_csv_col("./pulsatingSphere_tau05_step200_pv.csv", 3)
# pvdataRho = pvdataRho - rho0

# ax1.plot(pvDataX + 1.5, pvdataRho, ms=6, marker='^', fillstyle='none', c='r', lw=0, label='LBM BGK (ParaView Y=0 Z=0 step=10T)')
# ax1.plot(pvDataX + 1.5, pvdataRho, lw=2, linestyle='--', c='r', label='LBM BGK (ParaView Y=0 Z=0 step=10T)')

ax1.set_title(rf'$\tau={tau:.4f}$ $(\nu={vis:.4f})$ $T={int(1/f0)}$ ($step={int(step*f0)}T$)', loc='right')
ax1.legend(loc='best', frameon=False, fontsize=13)
ax1.set_xlim([rSphere, x[-1]])
ax1.set_ylim([-0.02, 0.02])
ax1.set_xlabel(r'$x$')
ax1.set_ylabel(r'$\rho^\prime$')